In [1]:
from ollama import Client
import pandas as pd

In [18]:
# Initialize the local LLM client
client = Client(host="http://localhost:11434")
model = 'tinyllama'

In [3]:
careplans_df = pd.read_csv('csv\careplans.csv')
medications_df = pd.read_csv('csv\medications.csv')
patients_df = pd.read_csv('csv\patients.csv')

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\aleci\AppData\Local\Temp\ipykernel_18524\174242808.py:1: SyntaxWarning: invalid escape sequence '\c'
  careplans_df = pd.read_csv('csv\careplans.csv')
C:\Users\aleci\AppData\Local\Temp\ipykernel_18524\174242808.py:2: SyntaxWarning: invalid escape sequence '\m'
  medications_df = pd.read_csv('csv\medications.csv')
C:\Users\aleci\AppData\Local\Temp\ipykernel_18524\174242808.py:3: SyntaxWarning: invalid escape sequence '\p'
  patients_df = pd.read_csv('csv\patients.csv')


In [4]:
cancer_careplans_df = careplans_df[careplans_df['DESCRIPTION'].str.contains("cancer", case=False, na=False)]

cancer_reasons = cancer_careplans_df['REASONDESCRIPTION'].unique()
print(f'Unique Cancer Reasons: {cancer_reasons}')

Unique Cancer Reasons: ['Secondary malignant neoplasm of colon' 'Malignant tumor of colon'
 'Neoplasm of prostate' 'Overlapping malignant neoplasm of colon'
 'Primary malignant neoplasm of colon']


In [5]:
# Predefined descriptions for the dataset's common cancers

clinic_cancer_info = [
    'Secondary malignant neoplasm of colon',
    'Malignant tumor of colon',
    'Neoplasm of prostate',
    'Overlapping malignant neoplasm of colon',
    'Primary malignant neoplasm of colon'
]


synonym_mapping = {
    'colon': [
        'Primary malignant neoplasm of colon',
        'Secondary malignant neoplasm of colon',
        'Overlapping malignant neoplasm of colon',
        'Malignant tumor of colon',
    ],
    'prostate': [
        'Neoplasm of prostate'
    ]
}

In [ ]:
filtered_medications = medications_df[medications_df['PATIENT'].isin(cancer_careplans_df['PATIENT'])]

result = filtered_medications[['DESCRIPTION', 'REASONDESCRIPTION']]

top_medications = result['DESCRIPTION'].value_counts().head(10).index

In [ ]:
filtered_patients = patients_df[patients_df['Id'].isin(cancer_careplans_df['PATIENT'])]

result = filtered_patients[['HEALTHCARE_EXPENSES','HEALTHCARE_COVERAGE']]

def five_number_summary(column):
    summary = {
        'min': column.min(),
        'q1': column.quantile(0.25),
        'median': column.median(),
        'q3': column.quantile(0.75),
        'max': column.max()
    }
    return summary

expenses_summary = five_number_summary(result['HEALTHCARE_EXPENSES'])
coverage_summary = five_number_summary(result['HEALTHCARE_COVERAGE'])

In [ ]:
#Zero Shot ICL

def generate_cancer_info(user_input):
    #Check for matches in synonym_mapping
    matched_cancers = []
    for synonym, cancers in synonym_mapping.items():
        if synonym.lower() in user_input.lower():
            matched_cancers.extend(cancers)

    #Build the initial response message
    if matched_cancers:
        message = f"We treat these specific types of cancers at our clinic: {', '.join(matched_cancers)}.\n"
    else:
        message = "We do not see many patients with this type of cancer here, but here is more about it:\n"

    #Generate descriptions based on matches or user input
    descriptions = []
    if matched_cancers:
        #Generate descriptions for each matched cancer
        for cancer in matched_cancers:
            prompt = f"""
            You are a compassionate healthcare assistant explaining cancer to families, including young children.
            Your goal is to provide a clear, empathetic, and simple explanation about the following cancer type: "{cancer}."

            Focus on:
            - Using language appropriate for children and families of cancer patients.
            - Providing essential details about the cancer type.
            - Conveying empathy and understanding.
            - Do not assume the user is the patient unless the input indicates so.
            - DO NOT INCLUDE *ACTIONS* IN THE RESPONSE.

            Cancer Type: {cancer}
            Response:
            """
            response = client.generate(
                model=model,
                prompt=prompt
            )
            descriptions.append(f"{cancer}: {response.response.strip()}")
    else:
        #If no matches, treat the user input as a broader query or unfamiliar cancer type
        prompt = f"""
        You are a compassionate healthcare assistant explaining cancer to families, including young children.
        Your goal is to provide a clear, empathetic, and simple explanation about the following question or type of cancer: "{user_input}."

        Focus on:
        - Identifying the type of cancer or question being asked based on the input.
        - Using language appropriate for children and families of cancer patients.
        - Providing essential details about the cancer or addressing the question.
        - Conveying empathy and understanding.
        - Do not assume the user is the patient unless the input indicates so.
        - DO NOT INCLUDE *ACTIONS* IN THE RESPONSE.

        User Input: {user_input}
        Response:
        """
        response = client.generate(
            model=model,
            prompt=prompt
        )
        descriptions.append(response.response.strip())

    result = f"{message}\n\n" + "\n\n".join(descriptions)
    return result



def generate_financial_summary(user_input):
    prompt = f"""
    You are a healthcare assistant helping families understand the financial aspects of cancer care.
    Your goal is to provide a clear, empathetic, and simple explanation about the following financial aspects: "{user_input}."

    Then use the following five-number summaries to explain average healthcare expenses and coverage for patients of our clinic in simple and empathetic terms:

    Healthcare Expenses:
    - Minimum: ${expenses_summary['min']:,.2f}
    - 25%ile: ${expenses_summary['q1']:,.2f}
    - Median: ${expenses_summary['median']:,.2f}
    - 75%ile: ${expenses_summary['q3']:,.2f}
    - Maximum: ${expenses_summary['max']:,.2f}

    Healthcare Coverage:
    - Minimum: ${coverage_summary['min']:,.2f}
    - 25%ile: ${coverage_summary['q1']:,.2f}
    - Median: ${coverage_summary['median']:,.2f}
    - 75%ile: ${coverage_summary['q3']:,.2f}
    - Maximum: ${coverage_summary['max']:,.2f}

    Explain what these numbers represent in a way that's easy to understand and compassionate.

    For example:
    - Explain how the range between Q1 and Q3 (the interquartile range) represents typical costs/coverage for most patients. But do not be overly technical.
    - Provide an overall view of the financial burden and how coverage helps.
    Response:
    """

    response = client.generate(
        model=model, 
        prompt=prompt
    )
    return response.response.strip()


In [ ]:
#Few Shot ICL

def generate_cancer_explanation_for_kids(user_input):
    prompt = f"""
    You are a compassionate healthcare assistant explaining cancer to families, including young children.
    Your goal is to provide a clear, empathetic, and simple explanation about the following: "{user_input}."

    Provide a simple, clear explanation that uses analogies or comparisons a child could understand. Be gentle and kind in your response.

    Examples:
    - What is chemotherapy?
      Response: Chemotherapy is a strong medicine that fights cancer cells. It's like a superhero team going after the bad guys in your body. Sometimes, it makes people feel tired, but it helps them get better.

    - What does a tumor mean?
      Response: A tumor is like a lump or bump in the body where cells grow too fast. Not all lumps and bumps are bad, and doctors can sometimes remove bad ones, like cleaning up a mess.

    - Do not assume the user is the patient unless the input indicates so.

    User Query: {user_input}
    Response:
    """
    response = client.generate(
        model=model,
        prompt=prompt
    )
    return response.response.strip()


def generate_medication_description_for_kids(user_input=None, top_medications=top_medications):

  if not user_input:
      #Case 1: No user input, iterate over top_medications
      medication_list_message = "Here are the most commonly prescribed medications for our cancer patients:\n"
      medication_list_message += "\n".join([f"{i+1}. {med}" for i, med in enumerate(top_medications)])
      medication_list_message += "\n\nFetching information for each medication...\n"

      descriptions = []
      for medication in top_medications:
          prompt = f"""
          You are a compassionate healthcare assistant explaining cancer to families, including young children.
          Please describe what the medication '{medication}' is commonly used for in simple, kind terms.
          Include analogies or examples to help make the explanation relatable.

          Example:
          - Medication: Ibuprofen
            Response: Ibuprofen is like a firefighter for your body. It reduces pain, swelling, and fever by calming down inflammation. It is commonly used for headaches, muscle aches, or minor injuries.
          
          Medication: {medication}
          Response:
          """
          response = client.generate(
              model=model,
              prompt=prompt
          )
          descriptions.append(f"{medication}: {response.response.strip()}")

      return medication_list_message + "\n\n" + "\n\n".join(descriptions)
  
  else:
      #Case 2: User input provided, identify medication and get information
      prompt = f"""
      You are a compassionate healthcare assistant explaining cancer to families, including young children.
      The user has asked about a medication in the following query: "{user_input}."
      
      Your goal is to:
      - Identify the medication name from the user input.
      - Provide a clear, empathetic, and simple explanation about the identified medication.
      - Use analogies or examples to make the response relatable, especially for children.
      
      Example:
      - Medication: Ibuprofen
        Response: Ibuprofen is like a firefighter for your body. It reduces pain, swelling, and fever by calming down inflammation. It is commonly used for headaches, muscle aches, or minor injuries.
      
      User Query: {user_input}
      Response:
      """
      response = client.generate(
          model=model,
          prompt=prompt
      )
      return response.response.strip()


In [ ]:
#Tree of Thought
def navigating_side_effects(user_input):
    prompt = f"""
    You are a compassionate healthcare assistant explaining cancer to families, including young children. 
    You have expertise in managing the side effects of cancer treatments.
    The user has asked: "{user_input}."

    Your task is to:
    1. Generate three possible responses or approaches to address the user's query.
        - Include remedies, strategies, or tools that are practical and empathetic.
        - Consider options such as medication, lifestyle adjustments, or complementary therapies.
    2. Evaluate the effectiveness, ease of implementation, and any potential risks of each response.
    3. Select the best response and explain why it is most suitable for the user's situation.

    Example:
    - Query: "What can I do about nausea from chemotherapy?"
      Option 1: "Anti-nausea medications prescribed by your doctor can help significantly. Be sure to discuss side effects with them."
        Why it's helpful: It offers a straightforward medical solution and encourages consultation with a professional.
      Option 2: "Ginger tea and small, frequent meals can sometimes reduce nausea naturally. Avoid greasy or spicy foods."
        Why it's helpful: It provides an easy-to-implement, natural remedy for users preferring non-medical approaches.
      Option 3: "Acupressure bands for motion sickness can sometimes ease chemotherapy-induced nausea."
        Why it's helpful: It introduces a complementary therapy for users open to trying alternative methods.
      Best Response: Option 1, because the user should first consult with their doctor for safe and effective anti-nausea treatment, though Options 2 and 3 may serve as complementary strategies.

    Now respond to the user's query:
    - Generate three possible responses.
    - Evaluate why each response could be helpful.
    - Select the best response and explain your reasoning. 
      Remember to advise that true medical advice should come from their doctor as well.
    """
    response = client.generate(
        model='llama3.2',
        prompt=prompt
    )
    return response.response.strip()

In [ ]:
#Chain of Thought
def understanding_new_diagnosis(user_input):
    prompt = f"""
    You are a compassionate healthcare assistant explaining cancer to families, including young children.
    You are tasked with helping patients and their families understand a new cancer diagnosis. 
    The user has asked: '{user_input}'.
    
    Break your response into clear, step-by-step explanations to address their concern thoughtfully and compassionately. 
    Make the explanation easy to follow for someone without medical expertise.

    Example:
    - Step 1: Introduce the staging system and explain its purpose in evaluating cancer.
    - Step 2: Provide specific details about the user's cancer stage (e.g., localized, spread).
    - Step 3: Describe common treatment options for this stage.
    - Step 4: Emphasize the importance of consulting their doctor for personalized advice.
    
    User Query: {user_input}
    Response:
    """
    response = client.generate(
        model=model,
        prompt=prompt
    )
    return response.response.strip()


In [ ]:
#Combined Methods
def generate_emotional_support_prompt(user_input):
    prompt = f"""
    You are a caring healthcare assistant designed to provide emotional support to the families of cancer patients.
    The user has shared: "{user_input}."

    Your task is to:

    1. Generate three possible responses:
       - A response tailored for a child.
       - A response tailored for a spouse.
       - A neutral response.

    2. Determine the likely role of the user based on their input (child, spouse, or neutral) using reasoning.
       - If the input mentions a parent (e.g., "mom" or "dad"), they are likely a child.
       - If the input mentions a partner (e.g., "husband" or "wife"), they are likely a spouse.
       - If the input is unclear or general, assume a neutral role.

    3. Select the best response based on empathy, clarity, and relevance, and return only the best response.

    Examples:
    - User: "I'm scared my dad won't get better."
      
      Option 1 (Child): "It's okay to feel scared. Your dad's doctors are doing everything they can to help him get better. You can make him happy by spending time with him and sharing your favorite stories."
        Why it's helpful: This reassures the child and provides actionable advice to comfort their parent.
      Option 2 (Spouse): "It's natural to feel scared, but remember, you're not alone in this journey. The medical team is working hard, and your support means the world to your husband."
        Why it's helpful: This validates the spouse's feelings and emphasizes their role as a source of support.
      Option 3 (Neutral): "It's normal to feel scared during tough times like these. Focus on the positive moments and trust in the care your loved one is receiving."
        Why it's helpful: This provides general comfort without assuming the user's specific relationship to the patient.
      Likely Role: Child
      Best Response: Option 1 (Child), because the input indicates the user is a child concerned about their dad's health, and the response provides reassurance and actionable advice.

    - User: "Why does my mom have to go to the hospital so much?"
      Option 1 (Child): "Hospitals are where doctors and nurses can give your mom the best care. Think of it as her superhero base where she gets stronger to fight the illness!"
        Why it's helpful: This uses a relatable analogy for a child to understand.
      Option 2 (Spouse): "Frequent hospital visits ensure your mom gets the best care possible. It's hard, but these visits are crucial for her treatment and recovery."
        Why it's helpful: This explains the necessity of hospital visits while showing empathy for the emotional toll on a spouse.
      Option 3 (Neutral): "Hospitals help people get better by providing the care and treatment they need. It might feel overwhelming, but it’s all to help your loved one recover."
        Why it's helpful: This provides a clear explanation that applies broadly.
      Likely Role: Child
      Best Response: Option 1 (Child), because the input indicates the user is a child seeking reassurance about their mom's hospital visits.

    - User: "I'm worried my husband seems so tired all the time during his treatment."
      
      Option 1 (Child): "Sometimes during treatment, people feel very tired, like they've been running a long race. Your dad might need extra rest, but he’s working hard to get better, and the doctors are helping him."
        Why it's helpful: This uses a relatable analogy and simple language, but it assumes the user is a child and doesn't directly address a spouse's emotional needs.
      Option 2 (Spouse): "It’s normal for your husband to feel tired during treatment. The fatigue is often a side effect of the medicine working to fight the illness. Encourage him to rest when he needs to, and let the care team know if his energy levels seem unusually low—they’re there to help."
        Why it's helpful: This response validates the spouse's concern, explains the fatigue, and provides practical advice while encouraging communication with the care team.
      Option 3 (Neutral): "Feeling tired is a common part of many treatments, as they take a lot of energy out of the body. Rest is essential during this time, and letting the care team know about these symptoms can help them provide the best support."
        Why it's helpful: This response is broadly applicable and provides useful information, but it lacks the personal connection needed for a spouse's specific emotional concern.
      Likely Role: Spouse
      Best Response: Option 2 (Spouse), because the input indicates the user is a spouse worried about their husband's fatigue. This response provides empathetic reassurance and actionable advice tailored to their role.
    
    Now respond to the user's query:
    - Generate three possible responses (child, spouse, neutral).
    - Determine the likely role of the user based on their input (child, spouse, or neutral).
    - Only return the best response—do not explain why it was chosen.
    """
    response = client.generate(
        model=model,
        prompt=prompt
    )
    return response.response.strip()

In [13]:
#MedLLama2

generate_cancer_info('Can you tell me about colon cancer?')

'We treat these specific types of cancers at our clinic: Primary malignant neoplasm of colon, Secondary malignant neoplasm of colon, Overlapping malignant neoplasm of colon, Malignant tumor of colon.\n\n\nPrimary malignant neoplasm of colon: Hello! I\'m here to help explain a type of cancer called primary malignant neoplasm of the colon, also known as colorectal cancer. It starts in the lining of the large intestine and can spread to other parts of the body. Treatment usually involves surgery and chemotherapy to kill any remaining cancer cells. While the symptoms might be uncomfortable for a family member or loved one, it\'s important to remember that colorectal cancer is a common type and many people are successfully treated each year. If you have any questions about the treatment options or how to support someone with this diagnosis, feel free to ask. Do you have anything else on your mind?\n\nThe goal of this response is to convey empathy while providing clear information in an age-

In [19]:
#TinyLLama

generate_cancer_info('What do you know about liver cancer?')

'We do not see many patients with this type of cancer here, but here is more about it:\n\n\nI can provide a revised version of your response that includes an example user input and relevant details about cancer, including:\n\nuser input: what do you know about live r cancer?\nresponse: live r cancer is a type of cancer that affects the blood cells in the body. The cancer usually starts with small lesions or tumors that gradually grow and spread, causing damage to vital organs and tissues. Treatment options may include chemotherapy, radiation therapy, or surgery depending on the extent and location of the disease. Cancer patients often experience side effects such as fatigue, nausea, and pain. I understand your concern, and please feel free to ask me any questions you have about living with cancer.'

In [14]:
#MedLLama2

generate_financial_summary("I'm worried about how much we will have to pay out of pocket for treatment.")

"Understanding healthcare expenses and insurance can be overwhelming, especially during a difficult time like this. It's important to know that we strive to make our services as affordable as possible. According to our records, the average out-of-pocket cost for cancer treatment at our clinic is around $197,599. This range includes all aspects of care including medication and hospitalization, but note that it can vary widely depending on individual circumstances. We understand that this information may be challenging to absorb right now, so please feel free to ask any questions or take a break if needed. What do you have concerns about?"

In [15]:
#MedLLama2

generate_cancer_explanation_for_kids("Why does my mom sometimes look yellow after treatment?")

"Oh no! That's because chemotherapy can make your Mom's skin a bit yellower than usual. It's like she's been drinking too much lemonade. Just like how sometimes you might turn orange from eating too many carrots, chemotherapy can change the color of some people's skin temporarily. But don't worry; it doesn't hurt and will go away eventually when her treatments are done. Can I help answer any other questions?"

In [20]:
#TinyLLama

generate_cancer_explanation_for_kids("What does the doctor mean when he says my brother has stage 2 cancer?")

"To respond to the question of what the doctor means when he says your brother has stage 2 cancer, provide a clear and simple explanation using analogies or comparison. Here's an example:\n\n- What is chemotherapy?\n   Response: Chemotherapy is a strong medicine that fights cancer cells. It's like a superhero team going after the bad guys in your body. Sometimes it can make people feel tired, but it helps them get better by killing off some of the cancer cells while keeping healthy cells intact. In simple terms, chemotherapy is an army trying to take down the enemy that's invading your body.\n\n- What does a tumor mean?\n   Response: A tumor is a lump or bulge in the body that doesn't belong there. It can be benign (meaning it won't harm you) or malignant (meaning it's cancerous). Sometimes, doctors remove bad tumors from your body to keep them from growing and spreading. In simple terms, tumors are a type of bulge or lump that doesn't belong there.\n\n- Do not assume the user is the p

In [16]:
#MedLLama2

generate_medication_description_for_kids("Why do I need to take Cytoxan?")

"Cytoxin is like a strong army of cells that go in your body and fight cancer cells by killing them. They help stop the growth of cancer cells, but it can also harm some healthy cells during this process. It's not an easy job, but they are very effective at getting rid of those bad cells.\n\nAsk follow-up questions if necessary to provide more information or clarification."

In [21]:
#TinyLLama

generate_medication_description_for_kids()

'Here are the most commonly prescribed medications for our cancer patients:\n1. insulin human  isophane 70 UNT/ML / Regular Insulin  Human 30 UNT/ML Injectable Suspension [Humulin]\n2. 24 HR Metformin hydrochloride 500 MG Extended Release Oral Tablet\n3. Hydrochlorothiazide 25 MG Oral Tablet\n4. amLODIPine 2.5 MG Oral Tablet\n5. Verapamil Hydrochloride 40 MG\n6. Digoxin 0.125 MG Oral Tablet\n7. Warfarin Sodium 5 MG Oral Tablet\n8. Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray\n9. Simvastatin 10 MG Oral Tablet\n10. Simvastatin 20 MG Oral Tablet\n\nFetching information for each medication...\n\n\ninsulin human  isophane 70 UNT/ML / Regular Insulin  Human 30 UNT/ML Injectable Suspension [Humulin]: "InsuliN Human ISOPHANE 70/30 (UNT/ML) is a commonly used medication for minor injuries and pain relief, especially in younger children. It\'s also known as \'insulin human\', which is the name of the injectable solution used to deliver insulin into your bloodstream."\n\nThis simple, relatable analo

In [22]:
#TinyLLama

navigating_side_effects("I find myself more confused throughout the day than I did before my treatment and I don't want to do.")

'Response 1:\n"I understand that feeling confused can be overwhelming, especially during treatment. Sometimes, fatigue and brain fog are common side effects of chemotherapy or other treatments. Have you considered talking to your doctor about potential adjustments to your medication regimen? They may be able to help find a better balance for you. In the meantime, getting enough rest, staying hydrated, and engaging in gentle exercises like yoga can also help alleviate some symptoms."\n\nWhy it\'s helpful: This response acknowledges the user\'s concern, suggests discussing medication adjustments with their doctor, and provides additional self-care strategies that are easy to implement.\n\nResponse 2:\n"Feeling confused all day is not normal, but sometimes it can be a sign of low blood sugar or dehydration. Have you eaten recently? Drinking plenty of water throughout the day can also help. Additionally, some people find value in using stress-reducing techniques like deep breathing exercis

In [17]:
#MedLLama2

understanding_new_diagnosis("The doctor just told me I have stage 1 Leukemia and I'm not sure what that means and what's going to happen to me.")

"Thank you for sharing this concern with me. Cancer staging helps doctors understand the extent of the cancer and plan treatment accordingly. For stage 1 Leukemia, your cancer is localized and hasn't spread beyond the bone marrow. This typically means treatment can be more straightforward. Some patients may only need treatment to manage their symptoms until the cancer goes away on its own. However, others might require stronger treatment. It's essential to consult your doctor for personalized advice based on your specific case. Does that help clarify things? What questions do you still have?"

In [23]:
#TinyLLama

generate_emotional_support_prompt("How much longer is my mom going to be doing treatment? She's always tired now.")

'User: "I\'m worried my husband seems so tired all the time during his treatment."\n\nBest Response: "It\'s okay to feel tired, and some patients do tend to feel sleepy or fatigued during their treatments. But as your husband\'s doctor continues to work on finding a cure for his illness, he\'s working hard to get better. The fatigue is often a side effect of the medicine working to fight the illness. Encourage him to rest when he needs to, and let the care team know if his energy levels seem unusually low—they\'re there to help."\n\nExample: "It\'s normal for your husband to feel tired during treatment, as his body is being pushed to its limits. But resting and letting the care team know about any unusual fatigue can help ensure he gets all the support needed during this time. Encourage him to talk to his doctor or nurse about any concerns he may have."\n\nBest Response: "Feeling tired is a common part of many treatment options, as they take a lot of energy out of the body. But resting